In [ ]:
import requests
from IPython.display import display, HTML

API_KEY = "11c5e50d678a364ea4652e61592afee2"
BASE_URL = "https://api.themoviedb.org/3"
IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"

def get_movies(endpoint, pages=5, params={}):
    all_results = []
    for page in range(1, pages + 1):
        url = f"{BASE_URL}/{endpoint}"
        params["api_key"] = API_KEY
        params["language"] = "pt-BR"
        params["page"] = page
        r = requests.get(url, params=params)
        if r.status_code == 200:
            results = r.json().get("results", [])
            all_results.extend(results)
        else:
            break
    return all_results

def get_movie_details(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {"api_key": API_KEY, "language": "pt-BR"}
    r = requests.get(url, params=params)
    return r.json()

def render_netflix_ui(sections):
    style = """
    <style>
      body { background:#141414; color:white; font-family:Arial; }
      .navbar { padding:20px; font-size:28px; font-weight:bold; color:#e50914; background:#000; }
      .section { margin:20px; }
      .section h3 { margin:10px 0; font-size:20px; }
      .row { display:flex; overflow-x:auto; scrollbar-width:none; }
      .row::-webkit-scrollbar { display:none; }
      .card {
        min-width:150px; height:220px; margin-right:10px; border-radius:6px;
        background-size:cover; background-position:center; flex-shrink:0;
        transition:transform 0.3s; cursor:pointer; position:relative;
      }
      .card:hover { transform:scale(1.08); }
      .card span {
        position:absolute; bottom:5px; left:5px; font-size:12px;
        background:rgba(0,0,0,0.6); padding:2px 5px; border-radius:3px;
      }
    </style>
    """
    html = '<div class="navbar">Netflix</div>'
    for section, items in sections.items():
        html += f'<div class="section"><h3>{section}</h3><div class="row">'
        for item in items:
            html += f"""
            <div class="card" style="background-image:url('{item['img']}');">
              <span>{item['title']}</span>
            </div>
            """
        html += '</div></div>'
    display(HTML(style + html))

In [ ]:
populares = get_movies("movie/popular")[:10]
melhores = get_movies("movie/top_rated")[:10]
lancamentos = get_movies("movie/now_playing")[:10]

movie_id = 299534

filme_base = get_movie_details(movie_id)
filme_nome = filme_base.get("title", "desconhecido")

similares = get_movies(f"movie/{movie_id}/similar")[:10]

In [ ]:
!pip install scikit-learn --quiet

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

base = get_movies("movie/popular")[:300]
textos = [f.get("overview") for f in base]

tfidf = TfidfVectorizer().fit_transform(textos)

indice_filme = 0
for i, f in enumerate(base):
  if f["title"] == filme_nome:
    indice_filme = i
    break


similaridades = cosine_similarity(tfidf[indice_filme], tfidf)[0]

indices = np.argsort(similaridades)[::-1][1:11]

similares = [
        {"title": base[i]["title"], "poster_path": base[i]["poster_path"]}
        for i in indices if base[i].get("poster_path")
    ]

In [ ]:
sections_data = {
    "Recomendados pra você":
     [
        {"title": m["title"], "img": IMAGE_BASE_URL + m["poster_path"]}
        for m in populares if m.get("poster_path")
     ],
    "Mais bem avaliados": [
        {"title": m["title"], "img": IMAGE_BASE_URL + m["poster_path"]}
        for m in melhores if m.get("poster_path")
    ],
    "Lançamentos": [
        {"title": m["title"], "img": IMAGE_BASE_URL + m["poster_path"]}
        for m in lancamentos if m.get("poster_path")
    ],
    f"Porque você assistiu {filme_nome}": [
        {"title": m["title"], "img": IMAGE_BASE_URL + m["poster_path"]}
        for m in similares if m.get("poster_path")
    ]
}

In [ ]:
render_netflix_ui(sections_data)